# Autocorrelation & spectral analysis — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def make_series(n=60, trend=0.08, amp=2.0, noise=0.25):
    t=np.arange(n)
    return t, 10 + trend*t + amp*np.sin(2*np.pi*t/12) + noise*np.random.randn(n)
def acf(x, max_lag):
    x=np.asarray(x)-np.mean(x); den=np.dot(x,x)
    return np.array([1.0]+[np.dot(x[:-k],x[k:])/den for k in range(1,max_lag+1)])
def moving_average(x,w):
    return np.convolve(x, np.ones(w)/w, mode='valid')
print('setup ok')

## Correlation in time and energy in frequency are two views of the same repeating structure
Autocorrelation asks which lags resemble the present; spectral analysis asks which cycles carry energy. These examples show lag correlation by hand, a seasonal ACF peak, a Fourier peak, a noise contrast, and trend leakage before detrending.

In [ ]:
# 1) hand ACF on [1,0,-1,0]: lag 2 is negative
y=np.array([1.,0.,-1.,0.]); r=[np.dot((y-y.mean())[:-k],(y-y.mean())[k:])/np.dot(y-y.mean(),y-y.mean()) for k in [1,2]]
plt.figure(figsize=(6,3)); plt.bar(['lag 1','lag 2'],r); plt.ylim(-0.7,0.2); plt.title('lag 2 compares opposite points')
assert np.allclose(r,[0.0,-0.5])

In [ ]:
# 2) seasonal signal: ACF peaks at its period
t=np.arange(72); y=np.sin(2*np.pi*t/12); vals=acf(y,24)
plt.figure(figsize=(6,3)); plt.stem(np.arange(25),vals); plt.title('period-12 seasonality gives lag-12 ACF peak')
assert vals[12] > 0.8 and vals[6] < -0.4

In [ ]:
# 3) Fourier periodogram: energy concentrates at frequency 1/12
t=np.arange(72); y=np.sin(2*np.pi*t/12); freq=np.fft.rfftfreq(len(y)); power=np.abs(np.fft.rfft(y))**2/len(y)
plt.figure(figsize=(6,3)); plt.stem(freq,power); plt.title('spectrum finds the 12-step cycle')
assert abs(freq[np.argmax(power)]-1/12)<1e-12

In [ ]:
# 4) white noise has no stable lag structure
y=np.random.randn(240); vals=acf(y,24); freq=np.fft.rfftfreq(len(y)); power=np.abs(np.fft.rfft(y))**2/len(y)
plt.figure(figsize=(6,3)); plt.plot(np.arange(25),vals,'o-',label='ACF'); plt.axhline(0,color='k'); plt.legend(); plt.title('noise ACF fluctuates around zero')
assert np.max(np.abs(vals[1:])) < 0.2

In [ ]:
# 5) detrending prevents low-frequency leakage from hiding seasonality
t=np.arange(120); y=0.05*t+np.sin(2*np.pi*t/12); coef=np.polyfit(t,y,1); yd=y-np.polyval(coef,t); freq=np.fft.rfftfreq(len(y)); p_raw=np.abs(np.fft.rfft(y))**2/len(y); p_det=np.abs(np.fft.rfft(yd))**2/len(y)
plt.figure(figsize=(6,3)); plt.plot(freq,p_raw,label='raw'); plt.plot(freq,p_det,label='detrended'); plt.xlim(0,0.2); plt.legend(); plt.title('remove trend before reading cycles')
assert p_raw[1] > p_det[1] and abs(freq[np.argmax(p_det[1:])+1]-1/12)<0.01